# Real-world Data Wrangling

In this project, you will apply the skills you acquired in the course to gather and wrangle real-world data with two datasets of your choice.

You will retrieve and extract the data, assess the data programmatically and visually, accross elements of data quality and structure, and implement a cleaning strategy for the data. You will then store the updated data into your selected database/data store, combine the data, and answer a research question with the datasets.

Throughout the process, you are expected to:

1. Explain your decisions towards methods used for gathering, assessing, cleaning, storing, and answering the research question
2. Write code comments so your code is more readable

## 1. Gather data

In this section, you will extract data using two different data gathering methods and combine the data. Use at least two different types of data-gathering methods.

### **1.1.** Problem Statement
In 2-4 sentences, explain the kind of problem you want to look at and the datasets you will be wrangling for this project.

I want to look into what type of aircraft are more involved in fatal accidents than other ones. In addition, I would like to see if year manufacturered has something to do with aircraft fatal accidents. I'm also curious if there are more fatal accidents in certain states and if that varies by aircraft type.

I will be using the FAA aircraft registry dataset and the NTSB accident database on Kaggle.

### **1.2.** Gather at least two datasets using two different data gathering methods

List of data gathering methods:

- Download data manually
- Programmatically downloading files
- Gather data by accessing APIs
- Gather and extract data from HTML files using BeautifulSoup
- Extract data from a SQL database

Each dataset must have at least two variables, and have greater than 500 data samples within each dataset.

For each dataset, briefly describe why you picked the dataset and the gathering method (2-3 full sentences), including the names and significance of the variables in the dataset. Show your work (e.g., if using an API to download the data, please include a snippet of your code). 

Load the dataset programmtically into this notebook.

In [1]:
import kagglehub
import os
import pandas as pd

#### **Dataset 1**

Type: CSV

Method: Kaggle Download

Dataset variables:

*   EventDate
*   State

In [26]:
#FILL IN 1st data gathering and loading method

# Download latest version
path = kagglehub.dataset_download("sainin/fatal-aviation-accidents-jan2010-feb2025")

accident_data = pd.read_csv(os.path.join(path, "a8f0c8d2-d8a5-4420-91b0-9033d2064f6fAviationData.csv"))

accident_data = accident_data[["N", "EventDate", "State"]]

accident_data.head()

,N,EventDate,State
0,N321BA,2025-02-06T16:20:00Z,Alaska
1,PK-LUV,2025-02-06T03:20:00Z,NaN
2,HC-CQY,2025-02-01T14:00:00Z,NaN
3,XA-UCI,2025-01-31T19:07:00Z,Pennsylvania
4,"N709PS, UNREG",2025-01-29T21:48:00Z,District of Columbia


#### Dataset 2

Type: CSV

Method: Download from FAA Website

Dataset variables:

*   Year MFR
*   Kit MFR

In [3]:
#FILL IN 2nd data gathering and loading method
registry_data = pd.read_csv("data-wrangling/aircraft-registry.csv")
registry_data = registry_data[["N-NUMBER", "YEAR MFR", "MFR MDL CODE"]]

model_data = pd.read_csv("data-wrangling/aircraft-models.csv")
model_data = model_data[["CODE", "MFR", "MODEL"]]

aircraft_data = registry_data.merge(model_data, left_on="MFR MDL CODE", right_on="CODE", how="left")

aircraft_data.head()

,N-NUMBER,YEAR MFR,MFR MDL CODE,CODE,MFR,MODEL
0,100,1940,7100510,7100510,PIPER,J3C-65
1,10000,,2130004,2130004,CIRRUS DESIGN CORP,SR22T
2,10001,1928,9601202,9601202,WACO,ASO
3,10004,,2072738,2072738,CESSNA,T182T
4,10006,1955,1152020,1152020,BEECH,D-45 (T-34B)


Optional data storing step: You may save your raw dataset files to the local data store before moving to the next step.

In [ ]:
#Optional: store the raw data in your local data store

## 2. Assess data

Assess the data according to data quality and tidiness metrics using the report below.

List **two** data quality issues and **two** tidiness issues. Assess each data issue visually **and** programmatically, then briefly describe the issue you find.  **Make sure you include justifications for the methods you use for the assessment.**

### Quality Issue 1:

In [37]:
accident_data.head()

accident_data[accident_data.duplicated(keep=False)]


,N,EventDate,State


In [36]:
accident_data = accident_data.drop_duplicates()

Issue and justification: There are duplicate values

### Quality Issue 2:

In [34]:
print(accident_data["State"].value_counts().sort_index())


State
Alabama                  57
Alaska                  146
Arizona                 118
Arkansas                 67
Atlantic Ocean           10
California              369
Caribbean Sea             2
Colorado                113
Connecticut              18
Delaware                  5
District of Columbia      2
Florida                 258
Georgia                 128
Gulf of Mexico            4
Hawaii                   26
Idaho                    67
Illinois                 59
Indiana                  59
Iowa                     40
Kansas                   40
Kentucky                 29
Louisiana                68
Maine                    19
Maryland                 20
Massachusetts            29
Michigan                 74
Minnesota                52
Mississippi              36
Missouri                 57
Montana                  45
Nebraska                 41
Nevada                   60
New Hampshire            11
New Jersey               35
New Mexico               57
New York      

In [41]:
invalid_states = ["Atlantic Ocean", "Caribbean Sea", "Gulf of Mexico"]

accident_data[accident_data["State"].isin(invalid_states)]

,N,EventDate,State


In [40]:
accident_data = accident_data[~accident_data["State"].isin(invalid_states)]


Issue and justification: There are values in the state columns that are bodies of water and not states.

### Tidiness Issue 1:

In [28]:
accident_data[accident_data["N"].str.contains(",", na=False)]

,N,EventDate,State
4,"N709PS, UNREG",2025-01-29T21:48:00Z,District of Columbia
63,"N313YK, N5287",2024-09-22T12:48:00Z,California
65,"N78074, N844CP",2024-09-16T09:46:00Z,Nevada
119,"VH-HQH, VH-HYQ",2024-07-25T06:10:00Z,NaN
137,"N6177J, N795LA",2024-07-11T14:07:00Z,Louisiana
...,...,...,...
4891,"G-JAST, G-MARX",2010-09-04T16:00:00Z,NaN
4934,"N7470C, N8829A",2010-08-04T18:20:00Z,Texas
5080,"N823AG, N788LL",2010-03-20T11:45:00Z,Florida
5121,"N8718L, N825BC, N2472W",2010-02-06T14:27:00Z,Colorado


In [29]:
# Each aircraft gets its own row
accident_data = accident_data.assign(
    N=accident_data["N"].str.split(",")
).explode("N")

# Clean up whitespace
accident_data["N"] = accident_data["N"].str.strip()

Issue and justification: In the case where there are multiple aircraft involved in an accident there are multiple tail numbers in the "N" column. In order count the accidents by type of aircraft I need to separate these into separate rows.

### Tidiness Issue 2: 

In [47]:
#FILL IN - Inspecting the dataframe visually
aircraft_data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 309586 entries, 0 to 309585
Data columns (total 6 columns):
 #   Column        Non-Null Count   Dtype 
---  ------        --------------   ----- 
 0   N-NUMBER      309586 non-null  object
 1   YEAR MFR      309586 non-null  object
 2   MFR MDL CODE  309586 non-null  object
 3   CODE          309586 non-null  object
 4   MFR           309586 non-null  object
 5   MODEL         309586 non-null  object
dtypes: object(6)
memory usage: 14.2+ MB


In [ ]:
#FILL IN - Inspecting the dataframe programmatically

Issue and justification: *FILL IN*

## 3. Clean data
Clean the data to solve the 4 issues corresponding to data quality and tidiness found in the assessing step. **Make sure you include justifications for your cleaning decisions.**

After the cleaning for each issue, please use **either** the visually or programatical method to validate the cleaning was succesful.

At this stage, you are also expected to remove variables that are unnecessary for your analysis and combine your datasets. Depending on your datasets, you may choose to perform variable combination and elimination before or after the cleaning stage. Your dataset must have **at least** 4 variables after combining the data.

In [ ]:
# FILL IN - Make copies of the datasets to ensure the raw dataframes 
# are not impacted

### **Quality Issue 1: FILL IN**

In [ ]:
# FILL IN - Apply the cleaning strategy

In [ ]:
# FILL IN - Validate the cleaning was successful

Justification: *FILL IN*

### **Quality Issue 2: FILL IN**

In [ ]:
#FILL IN - Apply the cleaning strategy

In [ ]:
#FILL IN - Validate the cleaning was successful

Justification: *FILL IN*

### **Tidiness Issue 1: FILL IN**

In [ ]:
#FILL IN - Apply the cleaning strategy

In [ ]:
#FILL IN - Validate the cleaning was successful

Justification: *FILL IN*

### **Tidiness Issue 2: FILL IN**

In [ ]:
#FILL IN - Apply the cleaning strategy

In [ ]:
#FILL IN - Validate the cleaning was successful

Justification: *FILL IN*

### **Remove unnecessary variables and combine datasets**

Depending on the datasets, you can also peform the combination before the cleaning steps.

In [ ]:
#FILL IN - Remove unnecessary variables and combine datasets

## 4. Update your data store
Update your local database/data store with the cleaned data, following best practices for storing your cleaned data:

- Must maintain different instances / versions of data (raw and cleaned data)
- Must name the dataset files informatively
- Ensure both the raw and cleaned data is saved to your database/data store

In [ ]:
#FILL IN - saving data

## 5. Answer the research question

### **5.1:** Define and answer the research question 
Going back to the problem statement in step 1, use the cleaned data to answer the question you raised. Produce **at least** two visualizations using the cleaned data and explain how they help you answer the question.

*Research question:* FILL IN from answer to Step 1

In [ ]:
#Visual 1 - FILL IN

*Answer to research question:* FILL IN

In [ ]:
#Visual 2 - FILL IN

*Answer to research question:* FILL IN

### **5.2:** Reflection
In 2-4 sentences, if you had more time to complete the project, what actions would you take? For example, which data quality and structural issues would you look into further, and what research questions would you further explore?

*Answer:* FILL IN